#### 4. Como forma de preparação para as próximas fases do curso, este desafio propõe uma reflexão prática sobre como a Ciência de Dados pode ser utilizada para antecipar a satisfação do cliente.

A partir da dor de negócio apresentada neste case, reflita sobre como um 
modelo preditivo poderia apoiar a empresa a prever o NPS antes da aplicação da 
pesquisa. Considere diferentes abordagens possíveis, como: 
- Um modelo de regressão, para estimar a nota de NPS em uma escala 
contínua; 
- Um modelo de classificação, para categorizar clientes, por exemplo, em 
satisfeitos e insatisfeitos.

#### Apresente uma proposta de modelo aplicada em **Python**, explicando de forma clara: 
- A definição da variável alvo; 
- A seleção e preparação das variáveis de entrada; 
- A lógica de separação dos dados (quando aplicável); 
- A escolha do modelo; 
- A forma de avaliação dos resultados; 
- E como essa solução poderia ser utilizada na prática pela empresa. 


In [1]:
#!pip install pandas matplotlib numpy seaborn scikit-learn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, average_precision_score,
                             classification_report, mean_absolute_error, mean_squared_error,
                             precision_score, r2_score, recall_score, roc_auc_score)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 2) CONFIGURAR: uma semente fixa torna o experimento reproduzível.
RANDOM_STATE = 42
TEST_SIZE = 0.20  # 80% treino | 20% teste, preservado até a avaliação final

dataFrameBronze = pd.read_csv("../data/raw/desafio_nps_fase_1.csv")

In [2]:
df_eda = dataFrameBronze.copy()
df_classificacao = dataFrameBronze.copy()

#### Atraso na Entrega tem uma influência grande no NPS

In [3]:
df_eda.select_dtypes(include='number').corr().round(4)

,customer_id,customer_age,customer_tenure_months,order_id,order_value,items_quantity,discount_value,payment_installments,delivery_time_days,delivery_delay_days,freight_value,delivery_attempts,customer_service_contacts,resolution_time_days,nps_score,repeat_purchase_30d,complaints_count,csat_internal_score
customer_id,1.0000,-0.0250,0.0244,1.0000,0.0116,-0.0079,-0.0079,0.0116,0.0136,-0.0181,-0.0035,0.0089,0.0112,-0.0001,0.0152,0.0025,-0.0120,0.0078
customer_age,-0.0250,1.0000,0.0299,-0.0250,0.0120,-0.0047,0.0070,-0.0508,0.0225,-0.0021,-0.0152,-0.0261,0.0327,0.0055,-0.0099,0.0090,0.0082,-0.0151
customer_tenure_months,0.0244,0.0299,1.0000,0.0244,0.0115,-0.0036,0.0153,0.0002,-0.0177,0.0012,-0.0175,-0.0260,-0.0090,-0.0052,-0.0097,0.0240,-0.0092,-0.0003
order_id,1.0000,-0.0250,0.0244,1.0000,0.0116,-0.0079,-0.0079,0.0116,0.0136,-0.0181,-0.0035,0.0089,0.0112,-0.0001,0.0152,0.0025,-0.0120,0.0078
order_value,0.0116,0.0120,0.0115,0.0116,1.0000,0.0040,0.0117,0.0423,-0.0219,0.0060,-0.0191,-0.0058,0.0328,-0.0143,0.0370,0.0234,0.0019,0.0008
items_quantity,-0.0079,-0.0047,-0.0036,-0.0079,0.0040,1.0000,0.0237,-0.0030,0.0203,-0.0146,-0.0114,-0.0202,-0.0146,0.0175,0.0115,0.0121,0.0250,-0.0056
discount_value,-0.0079,0.0070,0.0153,-0.0079,0.0117,0.0237,1.0000,0.0058,-0.0121,-0.0497,0.0206,0.0265,0.0161,-0.0172,0.0251,0.0145,0.0008,0.0401
payment_installments,0.0116,-0.0508,0.0002,0.0116,0.0423,-0.0030,0.0058,1.0000,-0.0079,-0.0266,-0.0006,0.0238,0.0255,0.0153,0.0237,-0.0107,-0.0019,0.0018
delivery_time_days,0.0136,0.0225,-0.0177,0.0136,-0.0219,0.0203,-0.0121,-0.0079,1.0000,-0.0066,-0.0175,-0.0196,0.0077,0.0212,0.0009,0.0061,-0.0090,-0.0248
delivery_delay_days,-0.0181,-0.0021,0.0012,-0.0181,0.0060,-0.0146,-0.0497,-0.0266,-0.0066,1.0000,0.0038,0.0052,-0.0242,-0.0153,-0.5973,-0.3040,0.1902,-0.4591


## Estratégia do estudo

Este notebook foi organizado para responder a duas perguntas de negócio:

1. Qual nota de NPS o cliente provavelmente dará?
2. Qual cliente está em maior risco de se tornar um detrator?

Para isso, construímos três abordagens complementares:

- modelo de regressão para prever a nota contínua de NPS;
- modelo de classificação para prever a categoria do cliente (promotor, passivo ou detrator);
- modelo binário para identificar risco de detrator.

A ideia é unir previsão analítica com ação operacional, permitindo que a empresa priorize clientes com maior chance de insatisfação antes que a pesquisa seja aplicada.

Importante: as variáveis preditoras utilizadas pelos modelos foram construídas a partir do conjunto de dados original, antes da divisão treino/teste. Ou seja, não houve criação de features dentro do próprio pipeline do modelo; todas as variáveis relevantes foram definidas no dataset base, mantendo o processo consistente e evitando vazamento de informação.

## Modelo preditivo — regressão de NPS

### Objetivo
A primeira abordagem busca estimar a nota contínua do NPS, permitindo que a empresa tenha uma previsão quantitativa antes da coleta oficial da pesquisa.

### Variável alvo
- `nps_score`: nota final do cliente no NPS.

### Variáveis de entrada
Foram utilizadas as variáveis relevantes para a experiência do cliente já presentes no dataset original, excluindo colunas que pudessem causar vazamento de informação, como indicadores que só são conhecidos após a jornada do cliente, como `repeat_purchase_30d` e `csat_internal_score`.

As variáveis foram selecionadas antes da separação dos dados, e não foram criadas dentro do próprio modelo. Isso mantém consistência entre a base analítica e a previsão, além de reduzir riscos de viés e de vazamento.

### Separação dos dados
Os dados foram divididos em treino e teste em proporção 80/20, preservando a distribuição da variável alvo por estratificação.

### Modelo
Foi aplicado um `RandomForestRegressor`, escolhido por sua capacidade de modelar relações não lineares e lidar com diferentes tipos de variáveis.

### Avaliação
- MAE: 1.33
- R²: 0.54

Esses resultados mostram que o modelo consegue capturar parte da variação da nota, mas ainda apresenta erro relevante em previsões pontuais. Portanto, a regressão é útil para estimar a nota, mas deve ser complementada por classificações operacionais.

In [4]:
# TARGET é o que queremos prever. FEATURES são os sinais usados para fazer a previsão.
TARGET = 'nps_score'
IDENTIFICADORES = ['customer_id', 'order_id']
VARIAVEIS_COM_VAZAMENTO = ['repeat_purchase_30d', 'csat_internal_score'] #Atributos que ocorrem depois da jornada do cliente, portanto não podem ser usados para prever o NPS.
FEATURES = [c for c in df_eda.columns if c not in [TARGET, *IDENTIFICADORES, *VARIAVEIS_COM_VAZAMENTO]]

# X recebe somente entradas e y recebe somente a resposta. Essa separação é padrão no scikit-learn.
X = df_eda[FEATURES].copy()
y = df_eda[TARGET].copy()

# Estratificação por decis: preserva a distribuição da nota contínua em treino e teste.
estratos_regressao = pd.qcut(y, q=10, duplicates='drop')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=estratos_regressao
)

print(f'Treino: {len(X_train):,} observações ({len(X_train)/len(X):.0%})')
print(f'Teste:  {len(X_test):,} observações ({len(X_test)/len(X):.0%})')
print('Features usadas:', FEATURES)
print('Excluídas por risco de vazamento:', VARIAVEIS_COM_VAZAMENTO)

Treino: 2,000 observações (80%)
Teste:  500 observações (20%)
Features usadas: ['customer_age', 'customer_region', 'customer_tenure_months', 'order_value', 'items_quantity', 'discount_value', 'payment_installments', 'delivery_time_days', 'delivery_delay_days', 'freight_value', 'delivery_attempts', 'customer_service_contacts', 'resolution_time_days', 'complaints_count']
Excluídas por risco de vazamento: ['repeat_purchase_30d', 'csat_internal_score']


In [5]:
variaveis_numericas = X.select_dtypes(include="number").columns
variaveis_categoricas = X.select_dtypes(exclude="number").columns

preprocessador = ColumnTransformer([
    (
        "numericas",
        SimpleImputer(strategy="median"),
        variaveis_numericas
    ),
    (
        "categoricas",
        Pipeline([
            ("preencher_nulos", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore"))
        ]),
        variaveis_categoricas
    )
])

modelo_nps = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=5
    ))
])

modelo_nps.fit(X_train, y_train)

previsoes_teste = modelo_nps.predict(X_test)

print(f"MAE: {mean_absolute_error(y_test, previsoes_teste):.2f}")
print(f"R²: {r2_score(y_test, previsoes_teste):.2f}")

MAE: 1.33
R²: 0.54


In [6]:
def classificar_nps(score):
    if score >= 9:
        return 'Promotor (9–10)'
    if score >= 7:
        return 'Passivo (7–8)'
    return 'Detrator (0–6)'


df_eda = dataFrameBronze.copy()
df_eda['categoria_nps'] = df_eda['nps_score'].apply(classificar_nps)

df_eda["nps_previsto"] = modelo_nps.predict(df_eda[FEATURES])

df_eda["risco_nps"] = np.where(
    df_eda["nps_previsto"] < 7,
    "Risco de detrator",
    "Sem risco de detrator"
)



In [7]:
df_detratores = df_eda[df_eda["categoria_nps"] == "Detrator (0–6)"].copy()

df_detratores["categoria_prevista"] = np.select(
    [
        df_detratores["nps_previsto"] >= 9,
        df_detratores["nps_previsto"] >= 7
    ],
    [
        "Promotor (9–10)",
        "Passivo (7–8)"
    ],
    default="Detrator (0–6)"
)



print(f"Quantidade de detratores reais: {len(df_detratores):,}")
print(f"Quantidade de detratores previstos: {len(df_detratores[df_detratores['categoria_prevista'] == 'Detrator (0–6)']):,} ")
print(f"Percentual de detratores previstos corretamente: {len(df_detratores[df_detratores['categoria_prevista'] == 'Detrator (0–6)']) / len(df_detratores):.2%} ")

Quantidade de detratores reais: 2,109
Quantidade de detratores previstos: 2,051 
Percentual de detratores previstos corretamente: 97.25% 


In [8]:
# Importância das variáveis para os modelos
# Esta célula deve ser lida como parte da etapa de interpretação do modelo, logo após o treinamento.

from sklearn.inspection import permutation_importance

# Importância para o modelo de regressão
resultado_regressao = permutation_importance(
    modelo_nps,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

importancia_regressao = pd.DataFrame({
    'feature': X_test.columns,
    'importancia_media': resultado_regressao.importances_mean,
    'importancia_desvio': resultado_regressao.importances_std
}).sort_values('importancia_media', ascending=False)

print('Top 10 variáveis mais importantes para a previsão da nota de NPS:')
display(importancia_regressao.head(10))



# Interpretação rápida
print('\nInterpretação:')
print('As variáveis com maior importância são as que mais afetam a experiência do cliente e, portanto, mais influenciam a nota e o risco de detrator.')
print('Esses resultados ajudam a explicar por que algumas variáveis têm maior peso do que outras na previsão do NPS.')


Top 10 variáveis mais importantes para a previsão da nota de NPS:


,feature,importancia_media,importancia_desvio
8,delivery_delay_days,0.611541,0.034316
13,complaints_count,0.226739,0.021157
11,customer_service_contacts,0.080662,0.017844
12,resolution_time_days,0.043811,0.007872
5,discount_value,0.005478,0.003338
3,order_value,0.002424,0.004250
10,delivery_attempts,0.001388,0.001685
0,customer_age,0.000644,0.001599
2,customer_tenure_months,0.000496,0.002284
1,customer_region,-0.000948,0.000776



Interpretação:
As variáveis com maior importância são as que mais afetam a experiência do cliente e, portanto, mais influenciam a nota e o risco de detrator.
Esses resultados ajudam a explicar por que algumas variáveis têm maior peso do que outras na previsão do NPS.


## Modelo preditivo — classificação NPS

### 1) Modelo de classificação por categoria
O segundo modelo foi construído para prever em qual categoria o cliente cairia:

- Detrator (0–6)
- Passivo (7–8)
- Promotor (9–10)

A variável alvo foi transformada em uma classificação categórica a partir do score real do cliente, usando a mesma lógica aplicada no dataset original. As variáveis preditoras foram mantidas como colunas já existentes na base, sem criação de atributos no momento do treinamento.

### 2) Modelo binário de risco de detrator
Além da classificação por categoria, foi testada uma abordagem binária para responder diretamente à pergunta de negócio: "qual cliente está em risco de virar detrator?"

A variável alvo foi `is_detrator`, com valor:

- 1 = detrator
- 0 = não detrator

Essa variável também foi definida no dataset original antes da separação dos dados, e o modelo usa as mesmas variáveis explicativas do conjunto base. Esse modelo é mais direto para ações operacionais, como campanhas de recuperação, atendimento prioritário e retenção.

**Nota de correção:** a primeira versão desta análise incluía `categoria_nps` como uma das features dos modelos de classificação. Como essa coluna é derivada diretamente do `nps_score` (a própria variável alvo), o modelo aprendia a "colar" a resposta em vez de generalizar a partir de sinais operacionais — por isso o resultado anterior mostrava accuracy de 1.00. A versão abaixo remove `categoria_nps` das features e reflete o desempenho real dos modelos.

### 3) Resultados

#### Classificação por categoria (Detrator / Passivo / Promotor)
- Accuracy: 0.82
- Detrator (0–6): precision 0.91, recall 0.92, f1 0.91
- Passivo (7–8): precision 0.31, recall 0.36, f1 0.33
- Promotor (9–10): precision 0.00, recall 0.00, f1 0.00

O modelo separa bem detratores do restante, mas praticamente não consegue distinguir passivos de promotores — esperado, já que essas duas classes juntas representam apenas ~15% da base e têm poucos exemplos de treino (56 e 22 no conjunto de teste).

#### Classificação binária (`is_detrator`)
- Accuracy: 0.80
- Classe detrator (1): precision 0.92, recall 0.79, f1 0.85
- Classe não detrator (0): precision 0.58, recall 0.81, f1 0.68

### 4) Interpretação
O modelo binário de risco de detrator é o mais robusto e o mais útil operacionalmente: ele erra menos ao identificar quem é detrator (recall 0.79, precision 0.92) do que ao tentar diferenciar as três categorias finas do NPS. Isso é coerente com a pergunta de negócio mais relevante para ação — "quem está em risco de insatisfação?" — que não exige acertar a nota exata, apenas o sinal de alerta.

A permutation importance confirma o padrão visto na EDA: `delivery_delay_days` e `complaints_count` dominam a previsão em todos os modelos, seguidos por `resolution_time_days` e `customer_service_contacts`. Ou seja, atraso na entrega e volume de reclamações são os fatores mais fortemente associados ao risco de detrator — não idade, região ou valor do pedido.

A combinação desses modelos oferece uma visão estratégica útil: a empresa pode estimar a nota, sinalizar risco de detrator com boa precisão, e priorizar ações para os casos mais críticos — mas a distinção fina entre passivo e promotor ainda precisa de mais dados ou variáveis para ser confiável.

In [9]:
df_classificacao['categoria_nps'] = df_classificacao['nps_score'].apply(classificar_nps)

TARGET = 'categoria_nps'   # ou 'is_detrator'
# 'categoria_nps' e 'nps_score' precisam ser excluídas das FEATURES: ambas derivam
# diretamente da nota real do cliente, então usá-las como entrada vazaria a própria
# resposta para o modelo (por isso a versão anterior tinha accuracy = 1.00).
FEATURES = [c for c in df_classificacao.columns if c not in ['nps_score', 'categoria_nps', 'customer_id', 'order_id', 'repeat_purchase_30d', 'csat_internal_score']]

X = df_classificacao[FEATURES].copy()
y = df_classificacao[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [10]:
df_classificacao['is_detrator'] = (df_classificacao['nps_score'] <= 6).astype(int)

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

variaveis_numericas = X.select_dtypes(include="number").columns
variaveis_categoricas = X.select_dtypes(exclude="number").columns

preprocessador = ColumnTransformer([
    ("numericas", SimpleImputer(strategy="median"), variaveis_numericas),
    ("categoricas", Pipeline([
        ("imputar", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), variaveis_categoricas)
])

modelo_classificacao = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ))
])

modelo_classificacao.fit(X_train, y_train)
predicoes = modelo_classificacao.predict(X_test)

print(classification_report(y_test, predicoes))
print(confusion_matrix(y_test, predicoes))

                 precision    recall  f1-score   support

 Detrator (0–6)       0.91      0.92      0.91       422
  Passivo (7–8)       0.31      0.36      0.33        56
Promotor (9–10)       0.00      0.00      0.00        22

       accuracy                           0.82       500
      macro avg       0.40      0.43      0.41       500
   weighted avg       0.80      0.82      0.81       500

[[389  31   2]
 [ 32  20   4]
 [  8  14   0]]


In [12]:
TARGET = 'is_detrator'
y = df_classificacao[TARGET].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

modelo_binario = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

modelo_binario.fit(X_train, y_train)
pred_binaria = modelo_binario.predict(X_test)

print(classification_report(y_test, pred_binaria))

              precision    recall  f1-score   support

           0       0.58      0.81      0.68       130
           1       0.92      0.79      0.85       370

    accuracy                           0.80       500
   macro avg       0.75      0.80      0.76       500
weighted avg       0.83      0.80      0.81       500



C:\Users\pedro\OneDrive\Documents\Pós-Fiap\Fase1\TechChallenge\Desafio\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [13]:
# Importância para o modelo de classificação binária de risco de detrator
resultado_binario = permutation_importance(
    modelo_binario,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

importancia_binaria = pd.DataFrame({
    'feature': X_test.columns,
    'importancia_media': resultado_binario.importances_mean,
    'importancia_desvio': resultado_binario.importances_std
}).sort_values('importancia_media', ascending=False)

print('\nTop 10 variáveis mais importantes para classificar risco de detrator:')
display(importancia_binaria.head(10))


# Interpretação rápida
print('\nInterpretação:')
print('As variáveis com maior importância são as que mais afetam a experiência do cliente e, portanto, mais influenciam a nota e o risco de detrator.')
print('Esses resultados ajudam a explicar por que algumas variáveis têm maior peso do que outras na previsão do NPS.')



Top 10 variáveis mais importantes para classificar risco de detrator:


,feature,importancia_media,importancia_desvio
13,complaints_count,0.1164,0.012060
8,delivery_delay_days,0.1110,0.015027
12,resolution_time_days,0.0228,0.007859
7,delivery_time_days,0.0054,0.001562
6,payment_installments,0.0050,0.003493
11,customer_service_contacts,0.0048,0.003124
5,discount_value,0.0040,0.003098
1,customer_region,0.0028,0.002993
9,freight_value,0.0024,0.003200
10,delivery_attempts,0.0020,0.003688



Interpretação:
As variáveis com maior importância são as que mais afetam a experiência do cliente e, portanto, mais influenciam a nota e o risco de detrator.
Esses resultados ajudam a explicar por que algumas variáveis têm maior peso do que outras na previsão do NPS.


## Conclusão

Este notebook mostrou que a previsão do NPS pode ser tratada com diferentes níveis de profundidade e utilidade para o negócio. O modelo de regressão foi útil para estimar a nota contínua e entender a intensidade da insatisfação, enquanto os modelos de classificação permitiram identificar o cliente por categoria e priorizar os casos com maior risco de detrator.

Os resultados indicam que a empresa consegue antecipar o comportamento dos clientes antes da aplicação da pesquisa, o que traz valor operacional e estratégico. Em particular, o modelo binário de risco de detrator é o mais aplicado para a tomada de decisão, pois responde diretamente à pergunta: 'quem merece atenção imediata?'

Em termos práticos, a solução proposta permite:

- prever a nota de NPS antes da pesquisa;
- agrupar clientes em promotores, passivos e detratores;
- identificar riscos de insatisfação antes que o problema se torne recorrente;
- direcionar ações preventivas de atendimento, retenção e melhoria de experiência.

Concluímos, portanto, que o modelo é uma ferramenta valiosa para apoio à decisão e para a gestão proativa da experiência do cliente. O próximo passo ideal seria validar esse comportamento em dados novos e em período real de operação, além de comparar outros algoritmos para garantir robustez em cenários reais.

In [14]:
# Aplicação dos modelos na base completa
# 1) Previsão da nota de NPS
# 2) Categorização da nota prevista
# 3) Probabilidade de ser detrator

# Garante que as colunas de features usadas no treino estejam presentes na base
base_modelo = df_eda.copy()

# Previsão da nota de NPS
base_modelo['nps_previsto'] = modelo_nps.predict(base_modelo[FEATURES])

# Classificação da nota prevista em promotor / passivo / detrator
base_modelo['categoria_prevista'] = np.select(
    [base_modelo['nps_previsto'] >= 9, base_modelo['nps_previsto'] >= 7],
    ['Promotor (9–10)', 'Passivo (7–8)'],
    default='Detrator (0–6)'
)

# Probabilidade de ser detrator
base_modelo['probabilidade_detrator'] = modelo_binario.predict_proba(base_modelo[FEATURES])[:, 1]

# Indicador binário de risco de detrator
base_modelo['risco_detrator'] = np.where(
    base_modelo['probabilidade_detrator'] >= 0.5,
    1,
    0
)

# Exibe as colunas adicionadas
base_modelo[['order_id', 'nps_score', 'nps_previsto', 'categoria_nps', 'categoria_prevista', 'probabilidade_detrator', 'risco_detrator']].head(10)


,order_id,nps_score,nps_previsto,categoria_nps,categoria_prevista,probabilidade_detrator,risco_detrator
0,50001,6.9,4.969804,Detrator (0–6),Detrator (0–6),0.398418,0
1,50002,2.4,2.534680,Detrator (0–6),Detrator (0–6),0.877477,1
2,50003,4.8,4.260442,Detrator (0–6),Detrator (0–6),0.877705,1
3,50004,5.9,4.507670,Detrator (0–6),Detrator (0–6),0.819036,1
4,50005,6.1,6.895961,Detrator (0–6),Detrator (0–6),0.121660,0
5,50006,0.9,1.093305,Detrator (0–6),Detrator (0–6),0.974811,1
6,50007,1.4,1.546015,Detrator (0–6),Detrator (0–6),0.950387,1
7,50008,0.0,1.286994,Detrator (0–6),Detrator (0–6),0.773660,1
8,50009,6.2,4.513853,Detrator (0–6),Detrator (0–6),0.799818,1
9,50010,2.7,4.604554,Detrator (0–6),Detrator (0–6),0.302270,0


In [15]:
# Resumo de negócio: clientes em risco de detrator

resumo_negocio = pd.DataFrame({
    'indicador': [
        'Clientes reais detratores',
        'Clientes previstos como detratores',
        'Percentual de detratores previstos',
        'Clientes em risco de detrator'
    ],
    'valor': [
        int((base_modelo['categoria_nps'] == 'Detrator (0–6)').sum()),
        int((base_modelo['categoria_prevista'] == 'Detrator (0–6)').sum()),
        round((base_modelo['categoria_prevista'] == 'Detrator (0–6)').mean() * 100, 2),
        int(base_modelo['risco_detrator'].sum())
    ]
})

print('Distribuição da categoria prevista:')
display(base_modelo['categoria_prevista'].value_counts().rename_axis('categoria_prevista').reset_index(name='quantidade'))

print('\nResumo de negócio:')
display(resumo_negocio)

print('\nTop clientes em risco de detrator:')
display(
    base_modelo[base_modelo['risco_detrator'] == 1][[
        'order_id', 'nps_score', 'nps_previsto', 'categoria_nps', 'categoria_prevista', 'probabilidade_detrator'
    ]]
    .sort_values('probabilidade_detrator', ascending=False)
    .head(10)
)


Distribuição da categoria prevista:


,categoria_prevista,quantidade
0,Detrator (0–6),2232
1,Passivo (7–8),264
2,Promotor (9–10),4



Resumo de negócio:


,indicador,valor
0,Clientes reais detratores,2109.00
1,Clientes previstos como detratores,2232.00
2,Percentual de detratores previstos,89.28
3,Clientes em risco de detrator,1596.00



Top clientes em risco de detrator:


,order_id,nps_score,nps_previsto,categoria_nps,categoria_prevista,probabilidade_detrator
1436,51437,0.2,0.536561,Detrator (0–6),Detrator (0–6),0.999840
328,50329,0.0,0.259555,Detrator (0–6),Detrator (0–6),0.999832
2355,52356,0.7,0.974184,Detrator (0–6),Detrator (0–6),0.999720
461,50462,0.0,0.804446,Detrator (0–6),Detrator (0–6),0.999640
1422,51423,0.0,0.217496,Detrator (0–6),Detrator (0–6),0.999611
1762,51763,0.0,0.658161,Detrator (0–6),Detrator (0–6),0.999593
1557,51558,0.0,0.315700,Detrator (0–6),Detrator (0–6),0.999555
954,50955,0.0,0.242378,Detrator (0–6),Detrator (0–6),0.999470
766,50767,0.0,0.275926,Detrator (0–6),Detrator (0–6),0.999455
595,50596,0.0,0.406549,Detrator (0–6),Detrator (0–6),0.999373
